In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
#Load the two cleaned datasets
import pandas as pd
df1 = pd.read_csv("/content/drive/MyDrive/AI chat bot/data set /NEW DATA SET /Cleaned data sets/ new_cleaned_data1.csv")
df2 = pd.read_csv("/content/drive/MyDrive/AI chat bot/data set /NEW DATA SET /Cleaned data sets/new_cleaned_data2.csv")

In [3]:
#remove spaces + lowercase to avoid errors
df1.columns = df1.columns.str.strip().str.lower()
df2.columns = df2.columns.str.strip().str.lower()

In [4]:
#Merge both datasets row-wise
wide_df = pd.concat([df1, df2], ignore_index=True)
wide_df.columns = wide_df.columns.str.strip().str.lower()


In [5]:
#Detect the animal column automatically
animal_col = "animalname"
if animal_col not in wide_df.columns:
    raise ValueError(f"'{animal_col}' not found. Columns: {wide_df.columns.tolist()}")

In [6]:
print(wide_df.columns.tolist())

['animalname', 'symptoms1', 'symptoms2', 'symptoms3', 'symptoms4', 'symptoms5', 'symptom_1', 'symptom_2', 'symptom_3', 'symptom_4']


In [7]:
#Detect ALL possible symptom columns
symptom_cols = [c for c in wide_df.columns if c.startswith("symptoms") or c.startswith("symptom_")]
if not symptom_cols:
    raise ValueError(f"No symptom columns found. Columns: {wide_df.columns.tolist()}")

print("Symptom columns detected:", symptom_cols)


Symptom columns detected: ['symptoms1', 'symptoms2', 'symptoms3', 'symptoms4', 'symptoms5', 'symptom_1', 'symptom_2', 'symptom_3', 'symptom_4']


In [8]:
print("Animal column:", animal_col)
print("Symptom columns found:", symptom_cols)

Animal column: animalname
Symptom columns found: ['symptoms1', 'symptoms2', 'symptoms3', 'symptoms4', 'symptoms5', 'symptom_1', 'symptom_2', 'symptom_3', 'symptom_4']


In [9]:
#Clean text values
wide_df[animal_col] = wide_df[animal_col].astype(str).str.strip().str.lower()
for c in symptom_cols:
    wide_df[c] = wide_df[c].astype(str).str.strip().str.lower()


In [10]:
#Create grouped
wide_df["case_id"] = range(1, len(wide_df) + 1)
print("Total original cases:", wide_df["case_id"].nunique())


Total original cases: 109


In [11]:
#one symptom per row, same case_id
long_df = wide_df.melt(
    id_vars=["case_id", animal_col],
    value_vars=symptom_cols,
    var_name="symptom_source",
    value_name="symptom"
)

In [12]:
#Remove empty
long_df["symptom"] = long_df["symptom"].astype(str).str.strip().str.lower()
long_df = long_df[~long_df["symptom"].isin(["", "none", "nan", "null"])]
long_df = long_df.dropna(subset=["symptom"])

In [13]:
#Remove duplicate symptoms inside same case
long_df = long_df.drop_duplicates(subset=["case_id", animal_col, "symptom"])

In [14]:
#Final column cleanup
final_df = long_df.rename(columns={animal_col: "animalname"})[["case_id", "animalname", "symptom"]]
final_df = final_df.sort_values(["case_id", "animalname", "symptom"]).reset_index(drop=True)

In [15]:
# Clean text first
final_df["symptom"] = ( final_df["symptom"].astype(str).str.strip().str.lower())

# Normalize symptoms
final_df["symptom"] = final_df["symptom"].replace({
    "vomitting": "vomiting",
    "loss of appettite": "loss of appetite",
    "loss of  appetite": "loss of appetite",
    "anoxeria": "anorexia",
    "lession on the skin": "lesion on the skin",
    "inflammed eye": "inflamed eye",
    "seizuers": "seizures",
    "chronic eye inflamation": "chronic eye inflammation",
    "tensemus": "tenesmus",
    "thrist and urination": "thirst and urination",
    "lossened teeth": "loosened teeth",
    "poor coat apperence": "poor coat appearance",
    "week legs": "weak legs",
    "week pulse": "weak pulse"
})



In [16]:
semantic_map = {
    # Appetite / not eating
    "appetite loss": "loss of appetite",
    "poor appetite": "loss of appetite",
    "anorexia": "loss of appetite",
    "no appp": "loss of appetite",

    # Diarrhea
    "watery stool": "diarrhea",
    "diarrhea with muscus": "diarrhea",

    # Breathing difficulty
    "difficulty breathing": "breathing difficulty",
    "labored breathing": "breathing difficulty",
    "respiratory distress": "breathing difficulty",
    "noisy breathing": "breathing difficulty",

    # Nasal discharge
    "runny nose": "nasal discharge",
    "greenish-yellow nasal discharge": "nasal discharge",

    # Low energy / lethargy
    "short term lethargy": "lethargy",
    "tiredness": "lethargy",
    "reduce energy": "lethargy",
    "depression": "lethargy",

    # Movement / lameness
    "limping": "lameness",
    "difficulty walking": "lameness",

    # Exercise intolerance
    "willnot run to jump": "unable to exercise",

    # Pain
    "pains": "pain",
    "painfull": "pain",
    "pain on leg": "pain",
    "pain on face": "pain",

    # Weakness
    "weak legs": "weakness",
    "weak pulse": "weakness",
    "weekness in the back legs": "weakness",
    "severe weekness and depression": "weakness",

    # Eyes
    "watering": "eye discharge",
    "inflamed eye": "eye inflammation",
    "chronic eye inflammation": "eye inflammation",

    # Skin
    "lesion on the skin": "skin lesions",
    "ulcerated skin": "skin lesions",
    "ulcers": "skin lesions",

    # Cough
    "strong cough": "coughing",

    # Lymph nodes
    "enlarged lymph nodes": "swollen lymph nodes",
    "enlarged lymph nodes or swelling": "swollen lymph nodes",

    # Swelling (general)
    "facial swelling": "swelling",
    "swelling on leg": "swelling",
    "swelling of joints": "swelling",
    "swelling of face or leg": "swelling",
}

final_df["symptom"] = final_df["symptom"].replace(semantic_map)

In [17]:
print(sorted(final_df["symptom"].unique()))
print("Total unique symptoms:", final_df["symptom"].nunique())

['abdominal pain', 'allergic reaction', 'anaemia', 'back pain', 'blindness', 'bloody drool', 'blue eye', 'breathing difficulty', 'broken bones', 'coughing', 'crusting of the skin', 'dandruff', 'darkened skin', 'dehydration', 'diarrhea', 'distended chest', 'eye and skin change', 'eye discharge', 'eye inflammation', 'fever', 'flatulence', 'foul breath', 'hair loss', 'hyperesthesia', 'jaundice', 'joint pain', 'lack of pigmentation', 'lameness', 'lesions in the nasal cavity', 'lesions on nose', 'lethargy', 'loosened teeth', 'loss of appetite', 'muscle pain', 'muscle stiffness', 'nasal discharge', 'neurologic abnormalities', 'nose bleeds', 'pain', 'pneumonia', 'poor coat appearance', 'rapid heart rate', 'seizures', 'skin lesions', 'sneezing', 'stiffness', 'straining', 'sudden death', 'swelling', 'swollen lymph nodes', 'tarry stool', 'tenesmus', 'thirst and urination', 'unable to eat', 'unable to exercise', 'upset stomach', 'vaginal discharge', 'vomiting', 'weakness', 'weight loss']
Total un

In [25]:
#Keep only dogs
dogs_only = final_df[final_df["animalname"] == "dog"].copy()

#Sort by symptom then case_id
dogs_sorted = dogs_only.sort_values(
    by=["symptom", "case_id"],
    kind="mergesort"   # stable sort (important)
)


In [26]:
dogs_sorted.to_csv(
    "/content/drive/MyDrive/AI chat bot/data set /NEW DATA SET /Cleaned data sets merged_dataset2.csv",
    index=False
)